# **Table Function Showcasing the Fairness Table**

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Literal, Optional, Union

import numpy as np
import pandas as pd


def make_fairness_paper_style_table(
    csv_path: Union[str, Path],
    *,
    decimals: int = 3,
    font_family: str = "Times New Roman",
    render: Literal["styler", "dataframe"] = "styler",
) -> Union[pd.io.formats.style.Styler, pd.DataFrame]:
    """
    Reads the provided CSV and returns a paper-style grouped table (Capacity, Safety, ...).

    CSV expected columns (your file matches this):
      Utility, Metric, Min, Max, Mean, StDev, Gini Coefficient, Alpha Fairness (α = 1, Normalized)

    - render="styler"  -> returns a pandas Styler with borders + bold group headers
    - render="dataframe" -> returns the prepared DataFrame (with group header rows inserted)
    """
    csv_path = Path(csv_path)
    df = pd.read_csv(csv_path)

    required = [
        "Utility",
        "Metric",
        "Min",
        "Max",
        "Mean",
        "StDev",
        "Gini Coefficient",
        "Alpha Fairness (α = 1, Normalized)",
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in CSV: {missing}")

    # Sort and build a "paper view" table with group header rows
    df = df[required].sort_values(["Utility", "Metric"]).reset_index(drop=True)

    out_rows = []
    section_row_flags = []  # True for group header rows
    numeric_cols = required[2:]

    for util, sub in df.groupby("Utility", sort=False):
        # group header row
        out_rows.append(
            {
                "Utility Metric": str(util),
                **{c: np.nan for c in numeric_cols},
            }
        )
        section_row_flags.append(True)

        # metric rows (indented label)
        for _, r in sub.iterrows():
            out_rows.append(
                {
                    "Utility Metric": f"  {r['Metric']}",
                    "Min": r["Min"],
                    "Max": r["Max"],
                    "Mean": r["Mean"],
                    "StDev": r["StDev"],
                    "Gini Coefficient": r["Gini Coefficient"],
                    "Alpha Fairness\n(α = 1, Normalized)": r["Alpha Fairness (α = 1, Normalized)"],
                }
            )
            section_row_flags.append(False)

    paper_df = pd.DataFrame(out_rows)

    # Rename numeric columns to match the screenshot-style header
    if "Alpha Fairness\n(α = 1, Normalized)" not in paper_df.columns:
        paper_df = paper_df.rename(
            columns={"Alpha Fairness (α = 1, Normalized)": "Alpha Fairness\n(α = 1, Normalized)"}
        )

    # Reorder columns exactly like the screenshot
    col_order = [
        "Utility Metric",
        "Min",
        "Max",
        "Mean",
        "StDev",
        "Gini Coefficient",
        "Alpha Fairness\n(α = 1, Normalized)",
    ]
    paper_df = paper_df[col_order]

    if render == "dataframe":
        return paper_df

    # ---- Styling (pandas Styler) ----
    styler = paper_df.style.hide(axis="index")

    # Format numbers; show blanks for section header rows (NaNs)
    fmt = {c: (lambda x, d=decimals: "" if pd.isna(x) else f"{x:.{d}f}") for c in col_order[1:]}
    styler = styler.format(fmt)

    # CSS styles to mimic paper table look
    table_styles = [
        # overall table look
        {"selector": "table", "props": [("border-collapse", "collapse"), ("font-family", font_family), ("font-size", "12pt")]},
        # header row: top border + bottom rule
        {"selector": "thead th", "props": [("border-top", "2px solid black"), ("border-bottom", "1.5px solid black"), ("font-weight", "normal"), ("text-align", "center"), ("padding", "6px 10px")]},
        # body cells: no grid, just spacing
        {"selector": "tbody td", "props": [("padding", "6px 10px")]},
        # bottom border
        {"selector": "tbody tr:last-child td", "props": [("border-bottom", "2px solid black")]},
        # left align first column
        {"selector": "tbody td:first-child", "props": [("text-align", "left")]},
        # right align numeric columns
        {"selector": "tbody td:not(:first-child)", "props": [("text-align", "right")]},
    ]
    styler = styler.set_table_styles(table_styles)

    # Bold group header rows + add a separator rule above each group header (except the first)
    def _style_sections(row):
        is_section = section_row_flags[row.name]
        if not is_section:
            return [""] * len(row)

        # bold group title, blank numeric cells already handled by formatter
        return [
            "font-weight: bold; border-top: 1.5px solid black;" if i == 0 else "border-top: 1.5px solid black;"
            for i in range(len(row))
        ]

    # Apply row-wise styling
    styler = styler.apply(_style_sections, axis=1)

    return styler

In [2]:

# -------------------------
# Example usage (your file)
# -------------------------

sty = make_fairness_paper_style_table(r"D:\Research Fellowship\NYC_Docked_Output_v2\FAIRNESS_RESULTS\NYC_DOCKED_fairness_results.csv")
sty  # in Jupyter this will display the styled table


Utility Metric,Min,Max,Mean,StDev,Gini Coefficient,"Alpha Fairness (α = 1, Normalized)"
Accessibility,,,,,,
Education institutions,0.000,16.800,3.039,3.018,0.514,145.205
Health services,0.000,44.200,5.779,6.508,0.556,107.493
Leisure activities,0.000,313.800,38.501,52.989,0.626,99.001
Public transit services,0.000,21.400,6.896,4.400,0.357,249.289
Total key destinations,0.200,397.200,58.220,64.633,0.535,118.911
Workplaces,0.000,32.800,4.006,5.279,0.625,99.104
Availability,,,,,,
Total vehicles available,0.000,557.174,49.026,46.552,0.428,41.873
Capacity,,,,,,
